In [1]:
print('this is for testing purpose')

this is for testing purpose


In [3]:
def churn_prediction_pipeline(file_path):
    import pandas as pd
    import numpy as np
    
    from sklearn.model_selection import train_test_split, GridSearchCV
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.metrics import accuracy_score, classification_report
    from sklearn.ensemble import RandomForestClassifier

    # -------------------------------
    # 1. Load Data
    # -------------------------------
    df = pd.read_csv(file_path)
    print("Data Loaded Successfully")
    
    # -------------------------------
    # 2. Basic Cleaning
    # -------------------------------
    df = df.drop_duplicates()
    df.columns = df.columns.str.strip()
    
    # Example: assume 'Churn' is target
    if 'Churn' not in df.columns:
        raise ValueError("Target column 'Churn' not found")

    # Convert target to numeric
    le = LabelEncoder()
    df['Churn'] = le.fit_transform(df['Churn'])

    # -------------------------------
    # 3. Feature & Target Split
    # -------------------------------
    X = df.drop('Churn', axis=1)
    y = df['Churn']

    # Identify column types
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_cols = X.select_dtypes(include=['object']).columns

    # -------------------------------
    # 4. Preprocessing Pipelines
    # -------------------------------
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', LabelEncoderWrapper())
    ])

    # Custom wrapper for LabelEncoder (since it doesn't work directly in pipeline)
    # -------------------------------
    # NOTE: Defined inline
    class LabelEncoderWrapper:
        def fit(self, X, y=None):
            self.encoders = {}
            for col in X.columns:
                le = LabelEncoder()
                X[col] = X[col].astype(str)
                self.encoders[col] = le.fit(X[col])
            return self

        def transform(self, X):
            X_copy = X.copy()
            for col in X_copy.columns:
                X_copy[col] = self.encoders[col].transform(X_copy[col].astype(str))
            return X_copy

        def fit_transform(self, X, y=None):
            return self.fit(X).transform(X)

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ])

    # -------------------------------
    # 5. Train-Test Split
    # -------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # -------------------------------
    # 6. Model + Pipeline
    # -------------------------------
    model_pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ])

    # -------------------------------
    # 7. Hyperparameter Tuning
    # -------------------------------
    param_grid = {
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5]
    }

    grid_search = GridSearchCV(
        model_pipeline,
        param_grid,
        cv=3,
        scoring='accuracy',
        n_jobs=-1
    )

    print("Training Model...")
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    print("Best Parameters:", grid_search.best_params_)

    # -------------------------------
    # 8. Evaluation
    # -------------------------------
    y_pred = best_model.predict(X_test)

    print("\nAccuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    # -------------------------------
    # 9. Feature Importance (Optional)
    # -------------------------------
    try:
        feature_names = list(numeric_cols) + list(categorical_cols)
        importances = best_model.named_steps['classifier'].feature_importances_
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances[:len(feature_names)]
        }).sort_values(by='Importance', ascending=False)

        print("\nTop Features:\n", importance_df.head())
    except:
        print("Feature importance not available")

    # -------------------------------
    # 10. Return Model
    # -------------------------------
    return best_model

In [4]:
def churn_prediction_pipeline(file_path):
    import pandas as pd
    import numpy as np
    
    from sklearn.model_selection import train_test_split, GridSearchCV
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.metrics import accuracy_score, classification_report
    from sklearn.ensemble import RandomForestClassifier

    # -------------------------------
    # 1. Load Data
    # -------------------------------
    df = pd.read_csv(file_path)
    print("Data Loaded Successfully")
    
    # -------------------------------
    # 2. Basic Cleaning
    # -------------------------------
    df = df.drop_duplicates()
    df.columns = df.columns.str.strip()
    
    # Example: assume 'Churn' is target
    if 'Churn' not in df.columns:
        raise ValueError("Target column 'Churn' not found")

    # Convert target to numeric
    le = LabelEncoder()
    df['Churn'] = le.fit_transform(df['Churn'])

    # -------------------------------
    # 3. Feature & Target Split
    # -------------------------------
    X = df.drop('Churn', axis=1)
    y = df['Churn']

    # Identify column types
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_cols = X.select_dtypes(include=['object']).columns

    # -------------------------------
    # 4. Preprocessing Pipelines
    # -------------------------------
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', LabelEncoderWrapper())
    ])

    # Custom wrapper for LabelEncoder (since it doesn't work directly in pipeline)
    # -------------------------------
    # NOTE: Defined inline
    class LabelEncoderWrapper:
        def fit(self, X, y=None):
            self.encoders = {}
            for col in X.columns:
                le = LabelEncoder()
                X[col] = X[col].astype(str)
                self.encoders[col] = le.fit(X[col])
            return self

        def transform(self, X):
            X_copy = X.copy()
            for col in X_copy.columns:
                X_copy[col] = self.encoders[col].transform(X_copy[col].astype(str))
            return X_copy

        def fit_transform(self, X, y=None):
            return self.fit(X).transform(X)

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ])

    # -------------------------------
    # 5. Train-Test Split
    # -------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # -------------------------------
    # 6. Model + Pipeline
    # -------------------------------
    model_pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ])

    # -------------------------------
    # 7. Hyperparameter Tuning
    # -------------------------------
    param_grid = {
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5]
    }

    grid_search = GridSearchCV(
        model_pipeline,
        param_grid,
        cv=3,
        scoring='accuracy',
        n_jobs=-1
    )

    print("Training Model...")
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    print("Best Parameters:", grid_search.best_params_)

    # -------------------------------
    # 8. Evaluation
    # -------------------------------
    y_pred = best_model.predict(X_test)

    print("\nAccuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    # -------------------------------
    # 9. Feature Importance (Optional)
    # -------------------------------
    try:
        feature_names = list(numeric_cols) + list(categorical_cols)
        importances = best_model.named_steps['classifier'].feature_importances_
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances[:len(feature_names)]
        }).sort_values(by='Importance', ascending=False)

        print("\nTop Features:\n", importance_df.head())
    except:
        print("Feature importance not available")

    # -------------------------------
    # 10. Return Model
    # -------------------------------
    return best_model

In [5]:
#this soli=utin is done churn prediction